# Phase 3 — Baseline RAG Pipeline and Evaluation

**Project:** Multi-Agent RAG Customer Support System

## Goal
Use the Phase 2 hybrid retriever to find relevant support-document chunks, generate a grounded answer, and save baseline evaluation results.

This is the baseline RAG system. LangGraph agents, validation/retries, feedback, and APIs come in later phases.

**Model:** `google/flan-t5-small`. It downloads on first use and requires internet. It can run on CPU but may be slow. No paid API key is required.

## Cell 1 — Locate the project root

Save this notebook in your project's `notebooks` folder. It searches parent folders for the Phase 1 data.

In [1]:
from pathlib import Path
import sys, json, re

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data" / "processed" / "knowledge_base_chunks.jsonl").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/processed/knowledge_base_chunks.jsonl. "
        "Save this notebook in the project's notebooks folder and verify Phase 1."
    )

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents
Python: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Scripts\python.exe


## Cell 2 — Install required libraries

Run this cell once. It installs the retriever and generation/evaluation dependencies.

In [2]:
# Install project dependencies without forcing a Transformers major-version upgrade.
# This avoids replacing the installed Transformers version with v5 just for this notebook.
%pip install sentence-transformers faiss-cpu rank-bm25 pandas numpy rouge-score


Note: you may need to restart the kernel to use updated packages.


## Cell 3 — Import libraries

In [3]:
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

print("Imports successful.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.
PyTorch: 2.14.1+cpu
CUDA available: False


## Cell 4 — Load Phase 1 chunks and Phase 2 FAISS index

The text and ID columns are detected from common names. If your Phase 1 notebook uses different column names, add them to the candidate lists.

In [4]:
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "knowledge_base_chunks.jsonl"
FAISS_PATH = PROJECT_ROOT / "data" / "vector_store" / "customer_support.faiss"

chunks = []
with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if line.strip():
            try:
                chunks.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_no}: {exc}") from exc

chunks_df = pd.DataFrame(chunks)
if chunks_df.empty:
    raise ValueError("No chunks loaded.")

TEXT_CANDIDATES = ["chunk_text", "text", "content", "page_content", "cleaned_text"]
TEXT_FIELD = next((c for c in TEXT_CANDIDATES if c in chunks_df.columns), None)
if TEXT_FIELD is None:
    raise KeyError(f"Text field not found. Available columns: {list(chunks_df.columns)}")

ID_CANDIDATES = ["chunk_id", "id", "chunkId"]
ID_FIELD = next((c for c in ID_CANDIDATES if c in chunks_df.columns), None)
if ID_FIELD is None:
    chunks_df["generated_chunk_id"] = [f"chunk_{i:04d}" for i in range(len(chunks_df))]
    ID_FIELD = "generated_chunk_id"

chunks_df[TEXT_FIELD] = chunks_df[TEXT_FIELD].fillna("").astype(str).str.strip()
chunks_df = chunks_df[chunks_df[TEXT_FIELD].ne("")].reset_index(drop=True)
texts = chunks_df[TEXT_FIELD].tolist()

if not FAISS_PATH.exists():
    raise FileNotFoundError(f"Missing {FAISS_PATH}. Run Phase 2 first.")
faiss_index = faiss.read_index(str(FAISS_PATH))

print("Chunks:", len(chunks_df))
print("Text field:", TEXT_FIELD)
print("ID field:", ID_FIELD)
print("FAISS vectors:", faiss_index.ntotal)
display(chunks_df.head(3))

Chunks: 18
Text field: text
ID field: chunk_id
FAISS vectors: 18


,chunk_id,document_id,title,category,source,chunk_index,text,word_count
0,FAQ001_CHUNK001,FAQ001,Track an order,orders,sample_faq,1,Customers can check an order's status from the...,45
1,FAQ002_CHUNK001,FAQ002,Cancel an order,orders,sample_faq,1,An order can usually be cancelled while it is ...,49
2,POL001_CHUNK001,POL001,Shipping and delivery policy,shipping,sample_policy,1,Standard delivery typically takes 3 to 7 busin...,51


## Cell 5 — Load embedding model and build BM25

In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

bm25_index = BM25Okapi([tokenize(t) for t in texts])
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("BM25 built for", len(texts), "chunks.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1764.93it/s]


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
BM25 built for 18 chunks.


## Cell 6 — Define semantic, keyword, and hybrid search

Hybrid search uses Reciprocal Rank Fusion (RRF) to combine rankings without assuming that semantic and BM25 scores share the same scale.

In [6]:
def semantic_search(query, top_k=5):
    vector = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    k = min(max(1, int(top_k)), len(texts))
    scores, indices = faiss_index.search(vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if 0 <= idx < len(chunks_df):
            row = chunks_df.iloc[int(idx)].to_dict()
            row["semantic_score"] = float(score)
            results.append(row)
    return results

def keyword_search(query, top_k=5):
    scores = bm25_index.get_scores(tokenize(query))
    ranked = np.argsort(scores)[::-1][:min(max(1, int(top_k)), len(texts))]
    results = []
    for idx in ranked:
        if scores[int(idx)] > 0:
            row = chunks_df.iloc[int(idx)].to_dict()
            row["bm25_score"] = float(scores[int(idx)])
            results.append(row)
    return results

def hybrid_search(query, top_k=5, candidate_k=10, rrf_k=60):
    semantic = semantic_search(query, min(candidate_k, len(texts)))
    keyword = keyword_search(query, min(candidate_k, len(texts)))
    fused, by_id = {}, {}
    for result_list in (semantic, keyword):
        for rank, item in enumerate(result_list, start=1):
            key = str(item[ID_FIELD])
            fused[key] = fused.get(key, 0.0) + 1.0 / (rrf_k + rank)
            by_id[key] = item
    ranked_ids = sorted(fused, key=fused.get, reverse=True)[:min(max(1, int(top_k)), len(fused))]
    results = []
    for rank, key in enumerate(ranked_ids, start=1):
        item = dict(by_id[key])
        item["rrf_score"] = float(fused[key])
        item["hybrid_rank"] = rank
        results.append(item)
    return results

print("Hybrid retriever ready.")

Hybrid retriever ready.


## Cell 7 — Load the language model

`google/flan-t5-small` is a small baseline encoder-decoder model. This notebook loads it with `AutoTokenizer` and `AutoModelForSeq2SeqLM`, rather than the `text2text-generation` pipeline. This is compatible with Transformers versions where that pipeline task has been removed.

The first run downloads model files. It can run on CPU but may be slow.


In [7]:
# No extra installation is needed here.
# We load FLAN-T5 directly with AutoModelForSeq2SeqLM instead of using
# the removed `text2text-generation` pipeline task in Transformers v5.
print("Using direct FLAN-T5 model loading for compatibility with Transformers v5.")


Using direct FLAN-T5 model loading for compatibility with Transformers v5.


In [8]:
GENERATION_MODEL_NAME = "google/flan-t5-small"

# Load FLAN-T5 as an encoder-decoder model. This avoids the
# `text2text-generation` pipeline task, which was removed in Transformers v5.
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

# Keep the same interface and return format expected by answer_question().
def generator(prompt, max_new_tokens=160, do_sample=False, **kwargs):
    inputs = tokenizer(
        str(prompt),
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    generation_options = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample
    }
    # Pass through a small, safe set of optional generation settings if supplied.
    for key in ("num_beams", "temperature", "top_p", "repetition_penalty"):
        if key in kwargs:
            generation_options[key] = kwargs[key]

    with torch.no_grad():
        output_ids = generation_model.generate(**inputs, **generation_options)

    answer_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return [{"generated_text": answer_text}]

print("Generation model loaded:", GENERATION_MODEL_NAME)
print("Device:", device)


c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tanish.Ranjan\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-age

Generation model loaded: google/flan-t5-small
Device: cpu


## Cell 8 — Build a prompt from retrieved evidence

In [9]:
def build_rag_prompt(question, retrieved_chunks, max_context_chars=4000):
    evidence = []
    used_chars = 0
    for item in retrieved_chunks:
        chunk_text = str(item[TEXT_FIELD]).strip()
        chunk_id = str(item[ID_FIELD])
        title = str(item.get("title", "Untitled"))
        block = f"[Source ID: {chunk_id} | Title: {title}]\n{chunk_text}\n"
        if used_chars + len(block) > max_context_chars:
            remaining = max_context_chars - used_chars
            if remaining > 100:
                evidence.append(block[:remaining])
            break
        evidence.append(block)
        used_chars += len(block)

    context = "\n".join(evidence)
    return (
        "You are a helpful customer support assistant. Answer using only the context. "
        "Do not invent policies or details. If the context does not answer the question, "
        "say you cannot confirm the answer from the available support documents. "
        "Keep the answer concise and mention relevant source IDs.\n\n"
        f"CONTEXT:\n{context}\nQUESTION: {question}\nANSWER:"
    )

print("Prompt builder ready.")

Prompt builder ready.


## Cell 9 — Create the baseline RAG answer function

In [10]:
def answer_question(question, top_k=4, max_new_tokens=160):
    retrieved = hybrid_search(question, top_k=top_k, candidate_k=max(10, top_k))
    if not retrieved:
        return {
            "question": question,
            "answer": "I could not find relevant information in the available support documents.",
            "sources": [],
            "retrieved_chunks": []
        }

    prompt = build_rag_prompt(question, retrieved)
    generated = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        truncation=True
    )[0]["generated_text"].strip()

    sources = [{
        "chunk_id": str(item[ID_FIELD]),
        "title": str(item.get("title", "Untitled")),
        "category": str(item.get("category", "")),
        "text": str(item[TEXT_FIELD])
    } for item in retrieved]

    return {
        "question": question,
        "answer": generated,
        "sources": sources,
        "retrieved_chunks": retrieved
    }

print("answer_question() is ready.")

answer_question() is ready.


## Cell 10 — Ask your first question

Change the question and run this cell again to try another scenario.

In [11]:
question = "How long do I have to return an item?"
result = answer_question(question, top_k=4)

print("QUESTION:", result["question"])
print("\\nANSWER:\\n", result["answer"])
print("\\nSOURCES:")
for source in result["sources"]:
    print(f"- {source['chunk_id']} | {source['title']} | {source['category']}")

QUESTION: How long do I have to return an item?
\nANSWER:\n 30 calendar days
\nSOURCES:
- POL002_CHUNK001 | Returns policy | returns
- FAQ002_CHUNK001 | Cancel an order | orders
- FAQ004_CHUNK001 | Request a refund | refunds
- FAQ005_CHUNK001 | Item arrived damaged | returns


## Cell 11 — Inspect the retrieved evidence

In [12]:
for rank, item in enumerate(result["retrieved_chunks"], start=1):
    print("=" * 80)
    print("Rank:", rank, "| Chunk ID:", item[ID_FIELD],
          "| RRF score:", round(item["rrf_score"], 6))
    print("Title:", item.get("title", "Untitled"))
    print(str(item[TEXT_FIELD])[:800])

Rank: 1 | Chunk ID: POL002_CHUNK001 | RRF score: 0.032522
Title: Returns policy
Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Customers should start a return from the order details page and follow the instructions provided. The return request must be approved before an item is sent back.
Rank: 2 | Chunk ID: FAQ002_CHUNK001 | RRF score: 0.032018
Title: Cancel an order
An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Cancel Order if the option is available. Orders that have already shipped cannot normally be cancelled; customers may request a return after delivery according to the returns policy.
Rank: 3 | Chunk ID: FAQ004_CHUNK001 | RRF score: 0.031754
Title: Request a refund
After a return is received and inspected, an approved refund 

## Cell 12 — Create a small evaluation dataset

These reference answers are starter references. Compare them with your sample support documents and edit if necessary.

In [13]:
evaluation_data = [
    {"question": "How can I track my order?",
     "reference_answer": "Customers can track an order using the order tracking option and their order details."},
    {"question": "What should I do if my item arrives damaged?",
     "reference_answer": "Customers should contact support and provide order details and information about the damage."},
    {"question": "How long do I have to return an item?",
     "reference_answer": "The return period is described in the return policy."},
    {"question": "What payment methods are accepted?",
     "reference_answer": "Accepted payment methods are listed in the payment policy."},
    {"question": "How can I protect my account?",
     "reference_answer": "Customers should follow account security practices and contact support if they suspect unauthorized access."}
]
eval_df = pd.DataFrame(evaluation_data)
display(eval_df)

,question,reference_answer
0,How can I track my order?,Customers can track an order using the order t...
1,What should I do if my item arrives damaged?,Customers should contact support and provide o...
2,How long do I have to return an item?,The return period is described in the return p...
3,What payment methods are accepted?,Accepted payment methods are listed in the pay...
4,How can I protect my account?,Customers should follow account security pract...


## Cell 13 — Generate answers for the evaluation set

This may take several minutes on CPU.

In [14]:
evaluation_rows = []
for i, item in eval_df.iterrows():
    print(f"Evaluating {i + 1}/{len(eval_df)}: {item['question']}")
    response = answer_question(item["question"], top_k=4)
    evaluation_rows.append({
        "question": item["question"],
        "reference_answer": item["reference_answer"],
        "generated_answer": response["answer"],
        "source_chunk_ids": ", ".join(s["chunk_id"] for s in response["sources"])
    })

baseline_eval_df = pd.DataFrame(evaluation_rows)
display(baseline_eval_df)

Evaluating 1/5: How can I track my order?
Evaluating 2/5: What should I do if my item arrives damaged?
Evaluating 3/5: How long do I have to return an item?
Evaluating 4/5: What payment methods are accepted?
Evaluating 5/5: How can I protect my account?


,question,reference_answer,generated_answer,source_chunk_ids
0,How can I track my order?,Customers can track an order using the order t...,[Source ID: FAQ001_CHUNK001 | Title: Track an ...,"FAQ001_CHUNK001, FAQ012_CHUNK001, FAQ003_CHUNK..."
1,What should I do if my item arrives damaged?,Customers should contact support and provide o...,Contact support within 7 days of delivery. Inc...,"FAQ005_CHUNK001, POL002_CHUNK001, POL005_CHUNK..."
2,How long do I have to return an item?,The return period is described in the return p...,30 calendar days,"POL002_CHUNK001, FAQ002_CHUNK001, FAQ004_CHUNK..."
3,What payment methods are accepted?,Accepted payment methods are listed in the pay...,"A payment can be declined by the bank, card is...","POL003_CHUNK001, FAQ011_CHUNK001, FAQ004_CHUNK..."
4,How can I protect my account?,Customers should follow account security pract...,Use a password that is not a password.,"FAQ008_CHUNK001, POL004_CHUNK001, FAQ011_CHUNK..."


## Cell 14 — Calculate ROUGE scores

ROUGE measures text overlap with a reference answer. It is a useful basic metric but does not prove factual correctness or grounding.

In [15]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
rouge_rows = []
for _, row in baseline_eval_df.iterrows():
    scores = rouge.score(str(row["reference_answer"]), str(row["generated_answer"]))
    rouge_rows.append({
        "question": row["question"],
        "rouge1_fmeasure": scores["rouge1"].fmeasure,
        "rougeL_fmeasure": scores["rougeL"].fmeasure
    })

rouge_df = pd.DataFrame(rouge_rows)
display(rouge_df)
print("Mean ROUGE-1 F1:", round(rouge_df["rouge1_fmeasure"].mean(), 4))
print("Mean ROUGE-L F1:", round(rouge_df["rougeL_fmeasure"].mean(), 4))

,question,rouge1_fmeasure,rougeL_fmeasure
0,How can I track my order?,0.272727,0.272727
1,What should I do if my item arrives damaged?,0.193548,0.161290
2,How long do I have to return an item?,0.000000,0.000000
3,What payment methods are accepted?,0.256410,0.153846
4,How can I protect my account?,0.000000,0.000000


Mean ROUGE-1 F1: 0.1445
Mean ROUGE-L F1: 0.1176


## Cell 15 — Manual quality review

Inspect whether the sources are relevant and whether each answer is supported by those sources. Fill in the blank columns after review.

In [16]:
manual_review_df = baseline_eval_df[
    ["question", "generated_answer", "source_chunk_ids"]
].copy()
manual_review_df["source_relevant"] = ""
manual_review_df["answer_supported"] = ""
manual_review_df["review_notes"] = ""
display(manual_review_df)

,question,generated_answer,source_chunk_ids,source_relevant,answer_supported,review_notes
0,How can I track my order?,[Source ID: FAQ001_CHUNK001 | Title: Track an ...,"FAQ001_CHUNK001, FAQ012_CHUNK001, FAQ003_CHUNK...",,,
1,What should I do if my item arrives damaged?,Contact support within 7 days of delivery. Inc...,"FAQ005_CHUNK001, POL002_CHUNK001, POL005_CHUNK...",,,
2,How long do I have to return an item?,30 calendar days,"POL002_CHUNK001, FAQ002_CHUNK001, FAQ004_CHUNK...",,,
3,What payment methods are accepted?,"A payment can be declined by the bank, card is...","POL003_CHUNK001, FAQ011_CHUNK001, FAQ004_CHUNK...",,,
4,How can I protect my account?,Use a password that is not a password.,"FAQ008_CHUNK001, POL004_CHUNK001, FAQ011_CHUNK...",,,


## Cell 16 — Save evaluation results

In [17]:
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = REPORT_DIR / "phase3_baseline_evaluation.csv"
ROUGE_PATH = REPORT_DIR / "phase3_rouge_scores.csv"
REVIEW_PATH = REPORT_DIR / "phase3_manual_review_template.csv"

baseline_eval_df.to_csv(BASELINE_PATH, index=False, encoding="utf-8")
rouge_df.to_csv(ROUGE_PATH, index=False, encoding="utf-8")
manual_review_df.to_csv(REVIEW_PATH, index=False, encoding="utf-8")

print("Saved:")
print(BASELINE_PATH)
print(ROUGE_PATH)
print(REVIEW_PATH)

Saved:
C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports\phase3_baseline_evaluation.csv
C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports\phase3_rouge_scores.csv
C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\reports\phase3_manual_review_template.csv


## Cell 17 — Final validation

In [18]:
assert len(chunks_df) > 0, "No chunks loaded."
assert faiss_index.ntotal > 0, "FAISS index is empty."
assert callable(answer_question), "RAG function not defined."
assert len(baseline_eval_df) == len(evaluation_data), "Evaluation row count mismatch."
assert BASELINE_PATH.exists() and ROUGE_PATH.exists() and REVIEW_PATH.exists()

print("Chunks loaded:", len(chunks_df))
print("FAISS vectors:", faiss_index.ntotal)
print("Evaluation questions:", len(baseline_eval_df))
print("Evaluation files saved successfully.")
print("\\nPHASE 3 BASELINE CHECK COMPLETE.")

Chunks loaded: 18
FAISS vectors: 18
Evaluation questions: 5
Evaluation files saved successfully.
\nPHASE 3 BASELINE CHECK COMPLETE.
